# Naumen CC — все запросы WFM Телесейлз-Сервис

База данных: `nccrep` (PostgreSQL, read-only).  
Файл источника: `backend/app/services/naumen_db.py`  

Перед запросами выполните ячейку **Подключение**.

In [ ]:
# ─── Подключение ─────────────────────────────────────────────────────────────
import psycopg2
import psycopg2.extras
import pandas as pd

conn_params = {
    'host':     'YOUR_HOST',
    'port':     5432,
    'database': 'nccrep',
    'user':     'YOUR_USER',
    'password': 'YOUR_PASSWORD',
}

def query(sql, params=None):
    with psycopg2.connect(**conn_params, cursor_factory=psycopg2.extras.RealDictCursor) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params or {})
            return pd.DataFrame([dict(r) for r in cur.fetchall()])

print('Готово. Используйте query(sql) для выполнения запросов.')

## 1. get_projects — Активные проекты (партнёры)
`backend/app/services/naumen_db.py` → `get_projects()`  
Источники: `mv_incoming_call_project`, `mv_outcoming_call_project`, `mv_partner`

In [ ]:
sql_projects = """
WITH active_projects AS (
    SELECT
        'incoming'  AS project_type,
        partneruuid AS partner_uuid,
        partnername AS partner_name,
        uuid        AS project_uuid
    FROM mv_incoming_call_project
    WHERE removed = false AND state = 'Активный'
    UNION ALL
    SELECT
        'outcoming' AS project_type,
        partneruuid AS partner_uuid,
        partnername AS partner_name,
        uuid        AS project_uuid
    FROM mv_outcoming_call_project
    WHERE removed = false AND state = 'Активный'
)
SELECT
    p.uuid                    AS customer_uuid,
    p.partnername             AS customer_name,
    p.partnertypetitle        AS customer_type,
    p.conditiontitle          AS customer_condition,
    p.responsiblemanagertitle AS responsible_manager,
    COUNT(ap.project_uuid)    AS active_projects_count,
    COUNT(ap.project_uuid) FILTER (WHERE ap.project_type = 'incoming')  AS active_incoming_count,
    COUNT(ap.project_uuid) FILTER (WHERE ap.project_type = 'outcoming') AS active_outcoming_count
FROM active_projects ap
LEFT JOIN mv_partner p ON p.uuid = ap.partner_uuid
WHERE p.removed = false AND p.conditiontitle = 'Активный'
GROUP BY p.uuid, p.partnername, p.partnertypetitle, p.conditiontitle, p.responsiblemanagertitle
ORDER BY p.partnername
"""

df_projects = query(sql_projects)
print(f'Проектов: {len(df_projects)}')
df_projects

## 2. get_queues — Очереди проекта
`backend/app/services/naumen_db.py` → `get_queues(partner_uuid)`  
Источник: `mv_incoming_call_project`

In [ ]:
PARTNER_UUID = 'ВСТАВЬТЕ_UUID_ПАРТНЁРА'  # customer_uuid из запроса выше

sql_queues = """
SELECT
    uuid                  AS queue_uuid,
    title                 AS name,
    datachannel           AS channel,
    servicelevelparameter AS target_sl,
    calllimit             AS answer_sec,
    state                 AS status
FROM mv_incoming_call_project
WHERE partneruuid = %(partner_uuid)s
  AND removed = false
  AND state = 'Активный'
ORDER BY title
"""

df_queues = query(sql_queues, {'partner_uuid': PARTNER_UUID})
print(f'Очередей: {len(df_queues)}')
df_queues

## 3. get_workload — Нагрузка по периодам (час/день)
`backend/app/services/naumen_db.py` → `get_workload(partner_uuid, begin, end, interval)`  
Источники: `queued_calls_ms`, `mv_incoming_call_project`, `call_legs`

In [ ]:
BEGIN_DATE = '2026-05-01'
END_DATE   = '2026-06-01'
INTERVAL   = 'hour'  # 'hour' или 'day'

trunc = 'hour' if INTERVAL == 'hour' else 'day'
sql_workload = f"""
SELECT
    date_trunc('{trunc}', que.enqueued_time)                   AS period_start,
    icp.title                                                    AS queue_name,
    COUNT(*)                                                     AS total,
    COUNT(*) FILTER (WHERE que.final_stage = 'operator')        AS handled,
    COUNT(*) FILTER (WHERE COALESCE(que.final_stage,'') <> 'operator') AS lost,
    ROUND(AVG(
        EXTRACT(EPOCH FROM (cl.ended - cl.connected))
    ) FILTER (
        WHERE que.final_stage = 'operator'
          AND cl.connected IS NOT NULL AND cl.ended IS NOT NULL
    )::numeric, 1)                                              AS avg_talk_sec,
    CASE WHEN MAX(icp.calllimit) > 0 THEN ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE que.final_stage = 'operator'
              AND que.unblocked_time_duration <= icp.calllimit * 1000
        ) / NULLIF(COUNT(*), 0), 2)
    ELSE NULL END                                               AS sl_percent
FROM queued_calls_ms que
JOIN mv_incoming_call_project icp ON icp.uuid = que.project_id
LEFT JOIN call_legs cl ON cl.session_id = que.session_id
                       AND cl.leg_id = que.next_leg_id
WHERE icp.partneruuid = %(partner_uuid)s
  AND icp.removed = false
  AND que.enqueued_time >= %(begin_date)s::timestamp
  AND que.enqueued_time <  %(end_date)s::timestamp
GROUP BY date_trunc('{trunc}', que.enqueued_time), icp.title
ORDER BY period_start, queue_name
"""

df_workload = query(sql_workload, {'partner_uuid': PARTNER_UUID, 'begin_date': BEGIN_DATE, 'end_date': END_DATE})
print(f'Строк: {len(df_workload)}')
df_workload.head(20)

## 4. get_operator_load — Нагрузка операторов за период
`backend/app/services/naumen_db.py` → `get_operator_load(partner_uuid, begin, end)`  
Источники: `queued_calls_ms`, `mv_incoming_call_project`, `call_legs`, `status_changes`, `mv_employee`

In [ ]:
sql_operator_load = """
WITH active_queues AS (
    SELECT uuid AS q_uuid, calllimit
    FROM mv_incoming_call_project
    WHERE partneruuid = %(partner_uuid)s AND removed = false
),
operator_calls AS (
    SELECT
        COALESCE(cl_op.dst_id, cl_op.dst_abonent,
                 cl_rd.dst_id, cl_rd.dst_abonent) AS login,
        COUNT(*) AS handled_calls,
        ROUND(AVG(
            COALESCE(
                EXTRACT(EPOCH FROM (cl_op.ended - cl_op.connected)),
                EXTRACT(EPOCH FROM (cl_rd.ended - cl_rd.connected))
            )
        )::numeric, 1) AS avg_talk_sec,
        ROUND(SUM(
            COALESCE(
                EXTRACT(EPOCH FROM (cl_op.ended - cl_op.connected)),
                EXTRACT(EPOCH FROM (cl_rd.ended - cl_rd.connected))
            )
        )::numeric, 0) AS total_talk_sec,
        ROUND(AVG(que.unblocked_time_duration / 1000.0) FILTER (
            WHERE que.final_stage = 'operator'
        )::numeric, 1) AS avg_answer_sec,
        ROUND(100.0 * COUNT(*) FILTER (
            WHERE que.final_stage = 'operator'
              AND que.unblocked_time_duration <= aq.calllimit * 1000
        ) / NULLIF(COUNT(*), 0)::numeric, 2) AS sl_percent
    FROM queued_calls_ms que
    JOIN active_queues aq ON aq.q_uuid = que.project_id
    LEFT JOIN call_legs cl_op ON cl_op.session_id = que.session_id
                              AND cl_op.leg_id = que.next_leg_id
                              AND que.final_stage = 'operator'
    LEFT JOIN call_legs cl_rd ON cl_rd.session_id = que.session_id
                              AND cl_rd.leg_id = que.next_leg_id
                              AND que.final_stage = 'redirect'
    WHERE que.enqueued_time >= %(begin_date)s::timestamp
      AND que.enqueued_time <  %(end_date)s::timestamp
      AND COALESCE(cl_op.dst_id, cl_op.dst_abonent,
                   cl_rd.dst_id, cl_rd.dst_abonent) IS NOT NULL
    GROUP BY COALESCE(cl_op.dst_id, cl_op.dst_abonent,
                      cl_rd.dst_id, cl_rd.dst_abonent)
),
status_summary AS (
    SELECT
        sc.login,
        ROUND(SUM(COALESCE(sc.duration, 0)) FILTER (
            WHERE lower(sc.status) NOT IN (
                'normal', 'ready', 'available', 'online', 'ringing', 'speaking',
                'inservice', 'ringing#voice', 'speaking#voice',
                'offline', 'logged_out', 'signedoff', 'loggedoff'
            )
        )::numeric, 0) AS idle_sec
    FROM status_changes sc
    WHERE sc.entered >= %(begin_date)s::timestamp
      AND sc.entered <  %(end_date)s::timestamp
    GROUP BY sc.login
)
SELECT
    oc.login          AS login,
    em.title          AS employee_name,
    em.post           AS position,
    oc.handled_calls  AS handled_calls,
    oc.avg_talk_sec   AS avg_talk_sec,
    oc.total_talk_sec AS total_talk_sec,
    oc.avg_answer_sec AS avg_answer_sec,
    oc.sl_percent     AS sl_percent,
    COALESCE(ss.idle_sec, 0) AS idle_sec
FROM operator_calls oc
LEFT JOIN mv_employee em ON em.login = oc.login
LEFT JOIN status_summary ss ON ss.login = oc.login
ORDER BY oc.handled_calls DESC
"""

df_opload = query(sql_operator_load, {'partner_uuid': PARTNER_UUID, 'begin_date': BEGIN_DATE, 'end_date': END_DATE})
print(f'Операторов: {len(df_opload)}')
df_opload.head(20)

## 5. get_status_summary — Суммарное время по статусам
`backend/app/services/naumen_db.py` → `get_status_summary(partner_uuid, begin, end)`  
Источники: `queued_calls_ms`, `mv_incoming_call_project`, `call_legs`, `status_changes`

In [ ]:
sql_status_summary = """
WITH active_queues AS (
    SELECT uuid AS q_uuid
    FROM mv_incoming_call_project
    WHERE partneruuid = %(partner_uuid)s AND removed = false
),
active_logins AS (
    SELECT DISTINCT COALESCE(cl_op.dst_id, cl_op.dst_abonent,
                             cl_rd.dst_id, cl_rd.dst_abonent) AS login
    FROM queued_calls_ms que
    JOIN active_queues aq ON aq.q_uuid = que.project_id
    LEFT JOIN call_legs cl_op ON cl_op.session_id = que.session_id
                              AND cl_op.leg_id = que.next_leg_id
                              AND que.final_stage = 'operator'
    LEFT JOIN call_legs cl_rd ON cl_rd.session_id = que.session_id
                              AND cl_rd.leg_id = que.next_leg_id
                              AND que.final_stage = 'redirect'
    WHERE que.enqueued_time >= %(begin_date)s::timestamp
      AND que.enqueued_time <  %(end_date)s::timestamp
      AND COALESCE(cl_op.dst_id, cl_op.dst_abonent,
                   cl_rd.dst_id, cl_rd.dst_abonent) IS NOT NULL
)
SELECT
    sc.login    AS login,
    sc.status   AS status,
    SUM(sc.duration) AS total_duration_sec,
    COUNT(*)    AS events_count
FROM status_changes sc
JOIN active_logins al ON al.login = sc.login
WHERE sc.entered >= %(begin_date)s::timestamp
  AND sc.entered <  %(end_date)s::timestamp
GROUP BY sc.login, sc.status
ORDER BY sc.login, total_duration_sec DESC
"""

df_status = query(sql_status_summary, {'partner_uuid': PARTNER_UUID, 'begin_date': BEGIN_DATE, 'end_date': END_DATE})
print(f'Строк (логин × статус): {len(df_status)}')
print('Все уникальные статусы:', sorted(df_status['status'].unique().tolist()))
df_status.head(30)

## 6. get_naumen_employees — Сотрудники из Naumen (по обработанным звонкам)
`backend/app/services/naumen_db.py` → `get_naumen_employees(partner_uuid, begin, end)`  
Источники: `queued_calls_ms`, `mv_incoming_call_project`, `call_legs`, `mv_employee`

In [ ]:
sql_employees = """
WITH active_queues AS (
    SELECT uuid AS q_uuid
    FROM mv_incoming_call_project
    WHERE partneruuid = %(partner_uuid)s AND removed = false
),
handled AS (
    SELECT
        COALESCE(cl.dst_id, cl.dst_abonent) AS login,
        COUNT(*)                             AS handled_calls,
        COUNT(DISTINCT aq.q_uuid)            AS queues_count
    FROM queued_calls_ms que
    JOIN active_queues aq ON aq.q_uuid = que.project_id
    LEFT JOIN call_legs cl ON cl.session_id = que.session_id
                           AND cl.leg_id = que.next_leg_id
    WHERE que.final_stage = 'operator'
      AND que.enqueued_time >= %(begin_date)s::timestamp
      AND que.enqueued_time <  %(end_date)s::timestamp
      AND COALESCE(cl.dst_id, cl.dst_abonent) IS NOT NULL
    GROUP BY COALESCE(cl.dst_id, cl.dst_abonent)
)
SELECT
    em.uuid           AS employee_uuid,
    em.login          AS login,
    em.title          AS employee_name,
    em.post           AS position,
    em.email          AS email,
    em.removed        AS is_removed,
    h.handled_calls   AS handled_calls,
    h.queues_count    AS queues_count
FROM handled h
JOIN mv_employee em ON em.login = h.login
ORDER BY em.title
"""

df_naumen_emp = query(sql_employees, {'partner_uuid': PARTNER_UUID, 'begin_date': BEGIN_DATE, 'end_date': END_DATE})
print(f'Сотрудников: {len(df_naumen_emp)}')
df_naumen_emp

## 7. sync_employees_for_partner — Сотрудники за 365 дней (для синхронизации)
`backend/app/services/naumen_db.py` → `sync_employees_for_partner(partner_uuid)`  
Тот же SQL что и выше, но `begin = today - 365d`, `end = today`

In [ ]:
from datetime import date, timedelta

begin_365 = (date.today() - timedelta(days=365)).isoformat()
end_today = date.today().isoformat()

df_sync = query(sql_employees, {'partner_uuid': PARTNER_UUID, 'begin_date': begin_365, 'end_date': end_today})
print(f'Сотрудников за 365 дней: {len(df_sync)}')
df_sync

## 8. get_active_logins_since — Активные логины с даты
`backend/app/services/naumen_db.py` → `get_active_logins_since(partner_uuid, cutoff_date)`  
Источники: `queued_calls_ms`, `mv_incoming_call_project`, `call_legs`

In [ ]:
CUTOFF_DATE = (date.today() - timedelta(days=30)).isoformat()  # активны за 30 дней

sql_active_logins = """
SELECT DISTINCT COALESCE(cl.dst_id, cl.dst_abonent) AS login
FROM queued_calls_ms que
JOIN mv_incoming_call_project icp ON icp.uuid = que.project_id
LEFT JOIN call_legs cl ON cl.session_id = que.session_id
                       AND cl.leg_id = que.next_leg_id
WHERE icp.partneruuid = %(partner_uuid)s
  AND icp.removed = false
  AND que.final_stage = 'operator'
  AND que.enqueued_time >= %(cutoff_date)s::timestamp
  AND COALESCE(cl.dst_id, cl.dst_abonent) IS NOT NULL
"""

df_active = query(sql_active_logins, {'partner_uuid': PARTNER_UUID, 'cutoff_date': CUTOFF_DATE})
print(f'Активных логинов с {CUTOFF_DATE}: {len(df_active)}')
df_active

## 9. get_operator_sessions — История сессий операторов по дням
`backend/app/services/naumen_db.py` → `get_operator_sessions(logins, begin, end)`  
Источник: `status_changes`

In [ ]:
# Замените на реальные логины операторов
TEST_LOGINS = ['operator_login_1', 'operator_login_2']

sql_sessions = """
WITH events AS (
    SELECT
        sc.login,
        sc.status,
        sc.entered,
        LEAD(sc.entered) OVER (
            PARTITION BY sc.login ORDER BY sc.entered
        ) AS next_entered
    FROM status_changes sc
    WHERE sc.login = ANY(%(logins)s)
      AND sc.entered >= %(begin_date)s::timestamp
      AND sc.entered <  %(end_date)s::timestamp
),
raw AS (
    SELECT
        login,
        DATE(entered) AS work_date,
        status,
        entered,
        GREATEST(0, LEAST(
            COALESCE(EXTRACT(EPOCH FROM (next_entered - entered)), 0),
            EXTRACT(EPOCH FROM (DATE_TRUNC('day', entered) + INTERVAL '1 day' - entered))
        )) AS duration,
        LAG(status) OVER (
            PARTITION BY login, DATE(entered) ORDER BY entered
        ) AS prev_status
    FROM events
)
SELECT
    login,
    work_date,
    MIN(entered) FILTER (WHERE status IN ('normal', 'available'))  AS first_login,
    COALESCE(
        MAX(entered) FILTER (WHERE status = 'offline'),
        MAX(entered + make_interval(secs => duration::float))
    ) AS last_logout,
    ROUND(SUM(duration)::numeric, 0) AS total_sec,
    ROUND(SUM(duration) FILTER (WHERE status IN ('normal', 'available'))::numeric, 0) AS normal_sec,
    ROUND(SUM(duration) FILTER (
        WHERE status NOT IN ('normal', 'available', 'offline')
    )::numeric, 0) AS non_normal_sec,
    ROUND(SUM(duration) FILTER (
        WHERE status != 'offline'
    )::numeric, 0) AS shift_sec,
    COUNT(*) FILTER (
        WHERE status NOT IN ('normal', 'available', 'offline')
          AND (prev_status IN ('normal', 'available') OR prev_status IS NULL)
    ) AS break_count,
    STRING_AGG(DISTINCT status, ', ' ORDER BY status) AS statuses_seen
FROM raw
GROUP BY login, work_date
ORDER BY login, work_date
"""

df_sessions = query(sql_sessions, {'logins': TEST_LOGINS, 'begin_date': BEGIN_DATE, 'end_date': END_DATE})
print(f'Строк: {len(df_sessions)}')
df_sessions

## 10. get_operator_timeline — Временная шкала событий оператора за день
`backend/app/services/naumen_db.py` → `get_operator_timeline(login, work_date)`  
Источник: `status_changes`

In [ ]:
LOGIN = 'operator_login_1'
WORK_DATE = '2026-06-10'

sql_timeline = """
WITH events AS (
    SELECT
        sc.login,
        sc.status,
        sc.entered,
        LEAD(sc.entered) OVER (
            PARTITION BY sc.login ORDER BY sc.entered
        ) AS next_entered
    FROM status_changes sc
    WHERE sc.login = %(login)s
      AND sc.entered >= %(work_date)s::timestamp
      AND sc.entered <  (%(work_date)s::date + INTERVAL '1 day')::timestamp
)
SELECT
    login,
    status,
    entered,
    GREATEST(0, LEAST(
        COALESCE(EXTRACT(EPOCH FROM (next_entered - entered)), 0),
        EXTRACT(EPOCH FROM (DATE_TRUNC('day', entered) + INTERVAL '1 day' - entered))
    )) AS duration_sec
FROM events
ORDER BY entered
"""

df_timeline = query(sql_timeline, {'login': LOGIN, 'work_date': WORK_DATE})
print(f'Событий за {WORK_DATE}: {len(df_timeline)}')
df_timeline

## 11. get_actual_operators_by_hour — Среднее факт. число операторов по часам
`backend/app/services/naumen_db.py` → `get_actual_operators_by_hour(logins, begin, end)`  
Источник: `status_changes`

In [ ]:
# Возьмите логины из запроса 6 или 8
ALL_LOGINS = df_naumen_emp['login'].tolist() if 'df_naumen_emp' in dir() else TEST_LOGINS

sql_actual_ops = """
WITH active_events AS (
    SELECT
        login,
        DATE(entered)             AS work_date,
        EXTRACT(HOUR FROM entered)::int AS hour_num
    FROM status_changes
    WHERE login = ANY(%(logins)s)
      AND status != 'offline'
      AND entered >= %(begin_date)s::timestamp
      AND entered <  %(end_date)s::timestamp
),
by_day_hour AS (
    SELECT
        work_date,
        hour_num,
        COUNT(DISTINCT login) AS operator_count
    FROM active_events
    GROUP BY work_date, hour_num
)
SELECT
    hour_num,
    ROUND(AVG(operator_count)::numeric, 1) AS avg_operators
FROM by_day_hour
GROUP BY hour_num
ORDER BY hour_num
"""

df_actual_ops = query(sql_actual_ops, {'logins': ALL_LOGINS, 'begin_date': BEGIN_DATE, 'end_date': END_DATE})
print(f'Часов: {len(df_actual_ops)}')
df_actual_ops

## 12. get_current_operators_for_project — Текущие статусы операторов (онлайн-мониторинг)
`backend/app/services/naumen_db.py` → `get_current_operators_for_project(partner_uuid)`  
Источники: `queued_calls_ms` (7 дней) + `status_changes` (48 часов) + `mv_employee`  

**Используется в LivePage для онлайн-мониторинга.**

In [ ]:
sql_current_ops = """
WITH project_logins AS (
    SELECT DISTINCT COALESCE(cl.dst_id, cl.dst_abonent) AS login
    FROM queued_calls_ms que
    JOIN mv_incoming_call_project icp ON icp.uuid = que.project_id
    LEFT JOIN call_legs cl ON cl.session_id = que.session_id
                           AND cl.leg_id = que.next_leg_id
    WHERE icp.partneruuid = %(partner_uuid)s
      AND icp.removed = false
      AND que.final_stage = 'operator'
      AND que.enqueued_time >= NOW() - INTERVAL '7 days'
      AND COALESCE(cl.dst_id, cl.dst_abonent) IS NOT NULL
),
latest_status AS (
    SELECT DISTINCT ON (sc.login)
        sc.login,
        sc.status,
        sc.entered
    FROM status_changes sc
    JOIN project_logins pl ON pl.login = sc.login
    WHERE sc.entered >= NOW() - INTERVAL '48 hours'
    ORDER BY sc.login, sc.entered DESC
)
SELECT
    ls.login         AS login,
    ls.status        AS status,
    ls.entered       AS entered,
    em.title         AS employee_name
FROM latest_status ls
LEFT JOIN mv_employee em ON em.login = ls.login
ORDER BY ls.status, ls.entered DESC
"""

df_current = query(sql_current_ops, {'partner_uuid': PARTNER_UUID})
print(f'Операторов в последних 48ч: {len(df_current)}')
print('Статусы:', df_current['status'].value_counts().to_dict())
df_current

## 13. get_current_online_operators — Последний статус для списка логинов
`backend/app/services/naumen_db.py` → `get_current_online_operators(logins)`  
Источник: `status_changes` (48 часов)

In [ ]:
# Используйте логины из предыдущего запроса
SOME_LOGINS = df_current['login'].tolist()[:10] if 'df_current' in dir() else TEST_LOGINS

sql_online = """
SELECT DISTINCT ON (login)
    login,
    status,
    entered
FROM status_changes
WHERE login = ANY(%(logins)s)
  AND entered >= NOW() - INTERVAL '48 hours'
ORDER BY login, entered DESC
"""

df_online = query(sql_online, {'logins': SOME_LOGINS})
print(f'Строк: {len(df_online)}')
df_online

## 14. get_operator_day_seconds — Суммарное время в статусах за день
`backend/app/services/naumen_db.py` → `get_operator_day_seconds(login, work_date)`  
Источник: `status_changes`

In [ ]:
sql_day_seconds = """
SELECT COALESCE(SUM(duration), 0) AS total_sec
FROM status_changes
WHERE login = %(login)s
  AND entered >= %(work_date)s::timestamp
  AND entered <  (%(work_date)s::date + INTERVAL '1 day')::timestamp
"""

df_day_sec = query(sql_day_seconds, {'login': LOGIN, 'work_date': WORK_DATE})
total_sec = df_day_sec['total_sec'][0]
print(f'Логин: {LOGIN}, дата: {WORK_DATE}')
print(f'Итого секунд: {total_sec} = {total_sec/3600:.2f} ч.')

## 15. test_connection — Проверка соединения
`backend/app/services/naumen_db.py` → `test_connection()`

In [ ]:
sql_ping = "SELECT 1 AS ok, NOW() AS server_time, version() AS pg_version"
df_ping = query(sql_ping)
print('Соединение успешно')
df_ping

## 16. Разведка статусов и профилей (диагностика)

Запросы для определения реальных статусов и профилей в вашей инсталляции Naumen.
Выполните при первом подключении, чтобы уточнить классификацию статусов в WFM.

**Custom\* статусы** (`Custom1`, `Custom2` и т.д.) — это ВСЕГДА пользовательские перерывы  
в Naumen CC. Они никогда не являются рабочими статусами — это архитектурное ограничение системы.

In [ ]:
# 16а. Все уникальные статусы за последние 90 дней (глобально, без фильтра по партнёру)
sql_all_statuses = """
SELECT
    status,
    COUNT(*)                                   AS events_count,
    COUNT(DISTINCT login)                       AS unique_logins,
    ROUND(AVG(duration)::numeric, 1)            AS avg_duration_sec,
    ROUND(SUM(duration)::numeric / 3600, 1)     AS total_hours
FROM status_changes
WHERE entered >= NOW() - INTERVAL '90 days'
GROUP BY status
ORDER BY events_count DESC
"""

df_all_statuses = query(sql_all_statuses)
print(f'Уникальных статусов: {len(df_all_statuses)}')
print('Список:', sorted(df_all_statuses['status'].tolist()))
print()

# Классифицируем автоматически
WORK   = {'normal','ready','available','ringing','speaking','inservice','ringing#voice','speaking#voice'}
OFFLN  = {'offline','logged_out','signedoff','loggedoff','disconnected'}

def classify(s):
    k = s.lower()
    if k in WORK:  return 'WORK'
    if k in OFFLN: return 'OFFLINE'
    if k.startswith('custom'): return 'PAUSE (custom)'
    return 'PAUSE'

df_all_statuses['class'] = df_all_statuses['status'].apply(classify)
df_all_statuses[['status','class','events_count','unique_logins','avg_duration_sec','total_hours']]

In [ ]:
# 16б. Статусы конкретного партнёра (используйте PARTNER_UUID из ячейки 2)
sql_partner_statuses = """
WITH project_logins AS (
    SELECT DISTINCT COALESCE(cl.dst_id, cl.dst_abonent) AS login
    FROM queued_calls_ms que
    JOIN mv_incoming_call_project icp ON icp.uuid = que.project_id
    LEFT JOIN call_legs cl ON cl.session_id = que.session_id
                           AND cl.leg_id = que.next_leg_id
    WHERE icp.partneruuid = %(partner_uuid)s
      AND icp.removed = false
      AND que.final_stage = 'operator'
      AND que.enqueued_time >= NOW() - INTERVAL '90 days'
      AND COALESCE(cl.dst_id, cl.dst_abonent) IS NOT NULL
)
SELECT
    sc.status,
    COUNT(*)                                   AS events_count,
    COUNT(DISTINCT sc.login)                   AS unique_logins,
    ROUND(AVG(sc.duration)::numeric, 1)        AS avg_duration_sec,
    ROUND(SUM(sc.duration)::numeric / 3600, 1) AS total_hours
FROM status_changes sc
JOIN project_logins pl ON pl.login = sc.login
WHERE sc.entered >= NOW() - INTERVAL '90 days'
GROUP BY sc.status
ORDER BY events_count DESC
"""

df_partner_st = query(sql_partner_statuses, {'partner_uuid': PARTNER_UUID})
df_partner_st['class'] = df_partner_st['status'].apply(classify)
print(f'Статусов партнёра: {len(df_partner_st)}')
df_partner_st

In [ ]:
# 16в. Профили и роли операторов (поле post из mv_employee)
sql_profiles = """
SELECT
    post         AS position,
    COUNT(*)     AS operator_count
FROM mv_employee
WHERE removed = false
GROUP BY post
ORDER BY operator_count DESC
"""

df_profiles = query(sql_profiles)
print('Профили/роли операторов:')
print(df_profiles.to_string(index=False))
print()

# Все колонки mv_employee для понимания схемы
df_emp_sample = query("SELECT * FROM mv_employee LIMIT 1")
print('Колонки mv_employee:', list(df_emp_sample.columns))